# ✈️ AeroLex — Building a Production-Grade RAG Pipeline

A hands-on build of a multi-stage Retrieval-Augmented Generation system over
**airline passenger-rights law** (EU 261/2004, US DOT 14 CFR 260, Delta Contracts
of Carriage).

**The pipeline:**
```
question
  └─▶ condense follow-up            (conversational memory)       [#8]
  └─▶ BM25 lexical  +  Gemini dense  (hybrid retrieval)           [#2,#3,#4]
  └─▶ reciprocal rank fusion         (merge the two)              [#5]
  └─▶ FLAIR feedback re-scoring      (learn from 👍/👎)            [#10]
  └─▶ cross-encoder re-ranking       (precise top-k)              [#6]
  └─▶ bounded reference expansion  (supporting evidence)        [#1]
  └─▶ grounded answer + [[n]] cites  (+ long-term profile)        [#7,#9]
```

**How to run:** pick the **`Python (aerolex RAG)`** kernel, then run top to bottom.
Feature #1 runs offline. Later stages need matching embeddings and installed
retrieval models; Gemini embedding/generation needs a `GEMINI_API_KEY`. You built each stage? The final section assembles them with the
`aerolex` library.


## 0 · Setup

In [ ]:
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'aerolex').is_dir() else Path.cwd().parent
sys.path.insert(0, str(root))
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # Windows: MKL + Torch OpenMP coexistence

import numpy as np
from aerolex import config
from aerolex.loaders import load_documents
from aerolex.chunking import ChunkCacheError, chunk_corpus, load_chunks, save_chunks
from aerolex.context import ReferenceExpander
from aerolex.store import InMemoryVectorStore

key = config.load_api_key()
print("GEMINI_API_KEY:", "✓ found" if key else "✗ not found — offline fallback mode")
print("gen model:", config.GEN_MODEL, "| embed model:", config.EMBED_MODEL, "| dim:", config.EMBED_DIM)

## 1 · Reference-aware Hierarchical Chunking · Feature #1 - Advitya

Legal text has a hierarchy: **document -> Article / section / RULE -> paragraph**.
Keep these boundaries, pack paragraphs/legal blocks into **at most 320 words**,
and fall back to sentences and then word windows for oversized blocks.
Up to **40 words of overlap are included in that budget**, never across sections.
These are word budgets, not exact model-token counts.

This is **structure-first chunking**, not an embedding-similarity semantic splitter.
Word/paragraph splitting is not universally better: evaluate the evidence needed
for your questions. Crucially, better boundaries alone cannot fetch another article.

```text
Baseline:        hybrid search -> rerank -> top-k passages -> answer
Reference-aware: hybrid search -> rerank -> seed sections
                 -> bounded outgoing + incoming references -> evidence -> answer
```

Each chunk's JSON has `references` (target chunk IDs) and `reference_mentions`
(verbatim mention, subclause locator, scope, and resolution status).
`Article 9(1)(b)` resolves to all parts of Article 9; it is not an exact subclause index.
References to other instruments stay unresolved unless explicitly mapped to this corpus.

**Why both directions?** Article 5 invokes Articles 7–9. Article 7 contains amounts
but does not point back to Article 5's exceptions. Incoming links discover invoking
provisions; they do not prove those provisions apply to the question.

The backend owns legal parsing (`references.py`). The reusable template below
only chunks documents, follows IDs, and assembles evidence (`context.py`).
Defaults: one hop, two incoming source sections per seed, 20 added chunks,
6,000 evidence words. Whole sections that do not fit are reported as omitted.
Complete seed sections that cannot fit raise an error instead of losing conditions.


In [ ]:
docs = load_documents()
for d in docs:
    print(f"{d.doc_id:16s} {d.doc_type:6s} {len(d.text):>7,d} chars  {d.title}")

try:
    chunks = load_chunks(documents=docs)
except (FileNotFoundError, ChunkCacheError) as error:
    print(f"Building current chunk cache: {error}")
    chunks = chunk_corpus(docs)
    save_chunks(chunks, documents=docs)

by_id = {c.chunk_id: c for c in chunks}
expander = ReferenceExpander(chunks)
art5 = next(c for c in chunks if c.section_id == ("eu261", "Article 5"))
art7 = next(c for c in chunks if c.section_id == ("eu261", "Article 7"))
assert all(len(c.text.split()) <= config.CHUNK_MAX_WORDS for c in chunks)
print(f"Loaded {len(chunks)} chunks; largest: {max(len(c.text.split()) for c in chunks)} words")


In [ ]:
import json
from dataclasses import asdict

print(json.dumps(asdict(art5), ensure_ascii=False, indent=2))
print("Resolved targets:", list(dict.fromkeys(by_id[cid].citation for cid in art5.references)))
external = next(m for c in chunks if c.section_id == ("eu261", "Article 2")
                for m in c.reference_mentions if m.status == "external")
print("Not guessed as local Article 2:", json.dumps(asdict(external), ensure_ascii=False))


### Two small, reusable exercises

1. Implement `fetch_referenced_chunks`: follow `chunk.references` through `by_id`,
   deduplicate IDs, and return the referenced chunks.
2. Implement `assemble_context`: call the shared expander on the selected seeds.
   Inspect its `chunks`, `provenance`, and `diagnostics`.

The solution notebook contains the completed functions. The demonstration after
these exercises uses the backend directly so Feature #1 can run offline even
before you fill in the functions.


In [ ]:
def fetch_referenced_chunks(chunk, by_id):
    return [by_id[cid] for cid in dict.fromkeys(chunk.references)]

def assemble_context(seed_chunks):
    return expander.expand(seed_chunks)

assert fetch_referenced_chunks(art5, by_id)
assert art5.chunk_id in assemble_context([art5]).seed_ids


In [ ]:
for seed in (art5, art7):
    context = expander.expand([seed])
    print("\nBaseline:", seed.citation)
    print("Expanded sections:", list(dict.fromkeys(c.section_id for c in context.chunks)))
    print("Discovery edges:", list(dict.fromkeys(
        (e.direction, e.source_section, e.target_section) for e in context.provenance)))
    print("Limits / missing evidence:", json.dumps(context.metadata()["diagnostics"], ensure_ascii=False))
    print("Evidence words:", context.metadata()["evidence_words"])

# Seed-controlled evidence coverage, NOT a claim about search or answer accuracy.
examples = [
    ("Cancellation compensation", art5, {"Article 5", "Article 7"}),
    ("Rerouting reduction", art7, {"Article 7", "Article 8"}),
    ("Care after cancellation", art5, {"Article 5", "Article 9"}),
]
print("\nQuestion | baseline coverage | expanded coverage | words")
for question, seed, required in examples:
    evidence = expander.expand([seed])
    present = {c.section_label for c in evidence.chunks}
    print(question, f"{len(required & {seed.section_label})}/{len(required)}",
          f"{len(required & present)}/{len(required)}", evidence.metadata()["evidence_words"], sep=" | ")


**Inspect before answering.** Try `expander.expand([art7], max_added_chunks=0)`
to see budget omissions, or `max_hops=2` to explore an additional reference hop.
Unresolved citations and budget/depth limits remain visible; no partial target
section is silently presented as complete.

Graph coverage is not legal correctness. Applicability may require provisions not
explicitly linked to a seed, facts missing from the question, or law outside these
PDFs. In particular, the EU261 PDF alone does not establish every later delay-
compensation judgment. We will expand evidence **after reranking** below, without
pruning the referenced conditions back down to the original top-k.


## 2 · Lexical Search with BM25  ·  Feature #2

Dense search blurs exact tokens, but statute codes must match **exactly**
(`§ 260.6`, `261/2004`). BM25 is a sparse bag-of-words scorer — the trick is a
**tokenizer that preserves those references** as single tokens.

### 🛠️ TODO: write `tokenize`

In [3]:
import re

# Keep words, the section sign §, and dotted/slashed numeric refs as single tokens.
_TOKEN = re.compile(r"[a-z]+|\u00A7|\d+(?:[./-]\d+)*")

def tokenize(text):
    return _TOKEN.findall(text.lower())

print(tokenize("See § 260.6 and Regulation 261/2004 under 14 CFR"))

['see', '§', '260.6', 'and', 'regulation', '261/2004', 'under', '14', 'cfr']


In [ ]:
from aerolex.bm25 import BM25Index

bm25 = BM25Index([c.chunk_id for c in chunks], [c.searchable_text for c in chunks])
for cid, score in bm25.search("§ 260.6 refund for a cancelled flight", top_k=3):
    print(f"{score:6.2f}  {by_id[cid].citation}")

## 3 · Dense Semantic Search + In-Memory Store  ·  Features #3 & #4

Gemini embeds every chunk into a vector; the whole corpus is one NumPy matrix.
Similarity is just a **normalized dot product** (cosine) — no vector database.

We load **pre-computed** embeddings from the cache so we don't embed 200+ chunks
live (the free tier is rate-limited).

In [ ]:
# Rebuild vectors once after rechunking; references themselves need no embedding calls.
if config.EMBEDDINGS_CACHE.exists():
    store = InMemoryVectorStore.load()
    store.validate_chunks(chunks)
elif key:
    from aerolex.embeddings import GeminiEmbedder
    matrix = GeminiEmbedder().embed_documents([c.searchable_text for c in chunks])
    store = InMemoryVectorStore.from_chunks(chunks, matrix)
    store.save()
else:
    raise RuntimeError("Feature #1 works offline. Later retrieval needs matching embeddings; "
                       "set GEMINI_API_KEY to build them, or supply a current validated cache.")
print("vector store:", len(store), "vectors x", store.matrix.shape[1], "dims")


### 🛠️ TODO: write `cosine_search` (normalized dot product top-k)

In [6]:
def cosine_search(query_vec, matrix, ids, top_k=5):
    q = query_vec / (np.linalg.norm(query_vec) or 1.0)
    M = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    scores = M @ q
    order = np.argsort(-scores)[:top_k]
    return [(ids[i], float(scores[i])) for i in order]

# sanity: a row is most similar to itself (cosine ≈ 1.0)
_hit = cosine_search(store.matrix[0], store.matrix, store.ids, top_k=1)[0]
print(_hit[0] == store.ids[0], round(_hit[1], 4))

True 1.0


In [ ]:
# Embed a real query (1 API call) and search. Needs a key; else we reuse a chunk vector.
query = "How much compensation for a long delayed flight under EU261?"
if key:
    from aerolex.embeddings import GeminiEmbedder
    qvec = GeminiEmbedder().embed_query(query)
else:
    print("(no key — demoing with an existing chunk vector as the 'query')")
    qvec = store.matrix[store.ids.index(art7.chunk_id)]

dense_hits = cosine_search(qvec, store.matrix, store.ids, top_k=5)
for cid, s in dense_hits:
    print(f"{s:5.3f}  {by_id[cid].citation}")

## 4 · Reciprocal Rank Fusion  ·  Feature #5

BM25 scores and cosine scores live on different scales, so we fuse by **rank**:
each list contributes `1 / (k + rank)` to an id's score (`k = 60`). Documents that
rank well in *either* retriever bubble up.

### 🛠️ TODO: write `rrf`

In [8]:
def rrf(rankings, k=60, top_k=10):
    scores = {}
    for ranked in rankings:
        for rank, cid in enumerate(ranked):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda kv: -kv[1])[:top_k]

assert rrf([["b", "a", "c"], ["b", "a", "d"]])[0][0] == "b"
print("rrf OK")

rrf OK


In [ ]:
bm25_ids = [cid for cid, _ in bm25.search(query, top_k=20)]
dense_ids = [cid for cid, _ in dense_hits]
fused = rrf([bm25_ids, dense_ids], k=config.RRF_K, top_k=10)
print("Fused top-5:")
for cid, s in fused[:5]:
    print(f"{s:.4f}  {by_id[cid].citation}")

## 5 · Cross-Encoder Re-Ranking  ·  Feature #6

First-stage retrieval scores query and document *independently*. A cross-encoder
reads them **together** (deep cross-attention) and is far more precise — so we run
it only over the small fused pool. (Model is pre-cached; runs locally.)

In [ ]:
from aerolex.rerank import CrossEncoderReranker

candidates = [(cid, by_id[cid].searchable_text) for cid, _ in fused]
reranked = CrossEncoderReranker().rerank(query, candidates, top_k=config.RERANK_TOP_K)
print("Re-ranked top chunks:")
for cid, s in reranked:
    print(f"{s:7.2f}  {by_id[cid].citation}")
# Reranking selects seeds; reference expansion assembles supporting evidence.
baseline_chunks = [by_id[cid] for cid, _ in reranked]
context = expander.expand(baseline_chunks)
final_chunks = context.chunks
print("Seed chunks:", len(baseline_chunks), "| evidence chunks:", len(final_chunks))
print("Evidence limits:", context.metadata()["diagnostics"])

## 6 · Grounded Answer with Inline Citations  ·  Feature #7

The model answers **only** from numbered passages and tags each claim with `[[n]]`.
A regex post-processor then validates those tags against the real passages and
**drops hallucinated ones** — the difference between "looks cited" and "is cited".

### 🛠️ TODO: write `resolve_citations`

In [ ]:
CITE = re.compile(r"\[\[(\d+)\]\]")

def resolve_citations(answer, index):
    used = []
    def keep(m):
        n = int(m.group(1))
        if n in index:
            if n not in used:
                used.append(n)
            return m.group(0)
        return ""                      # drop a citation to a non-existent passage
    cleaned = CITE.sub(keep, answer)
    return cleaned, [f"[[{n}]] {index[n]}" for n in sorted(used)]

_c, _r = resolve_citations("A [[1]] and B [[9]].", {1: "EU261 Art 7", 2: "EU261 Art 5"})
print(_c, "|", _r)

In [ ]:
from aerolex.generate import Generator

result = Generator().generate(query, final_chunks, context_diagnostics=context.metadata())
print("OFFLINE" if result["offline"] else "GEMINI", "answer:\n")
print(result["answer"])
print("\nReferences:")
for r in result["references"]:
    print("  ", r)

## 7 · Conversational Memory  ·  Feature #8

Follow-ups like *"what about in the US?"* are meaningless to a retriever alone. We
**condense** the chat history + follow-up into a standalone query before searching.

In [ ]:
from aerolex.memory import ConversationMemory

mem = ConversationMemory()
mem.add_user(query)
mem.add_assistant(result["answer"][:160])
print("follow-up : what about in the US?")
print("standalone:", mem.condense("what about in the US?"))

## 8 · Long-Term Persistence  ·  Feature #9

Facts a user states persist across sessions in SQLite and can be injected into
future prompts.

In [ ]:
from aerolex.persistence import ProfileStore

prof = ProfileStore()
prof.remember("demo_user", "home_airport", "JFK")
prof.remember("demo_user", "status", "Delta Platinum Medallion")
print(prof.context_block("demo_user"))

## 9 · Adaptive Feedback — FLAIR  ·  Feature #10

We log `(query, chunk, 👍/👎)` triplets and nudge future rankings by **query-weighted
votes**: a past vote counts only when the new query is similar to the one it was
cast for — so feedback generalizes to unseen-but-similar questions.

In [ ]:
from aerolex.feedback import FeedbackStore

fb = FeedbackStore()
top_id = reranked[0][0]
fb.record(query, qvec, top_id, +1)                 # 👍 the best chunk
ranked = [(cid, s) for cid, s in reranked]
adjusted = fb.apply(ranked, qvec)
print("before:", [by_id[c].section_label for c, _ in ranked])
print("after :", [by_id[c].section_label for c, _ in adjusted])

## 10 · The Full Pipeline

Now assemble every stage with the `aerolex` library — the productionized version of
what you just built — and ask a question end to end.

In [ ]:
from aerolex.pipeline import RagPipeline

if key:
    from aerolex.embeddings import GeminiEmbedder
    pipe = RagPipeline(
        chunks=chunks,
        store=store,
        bm25=bm25,
        embedder=GeminiEmbedder(),
        reranker=CrossEncoderReranker(),
        generator=Generator(),
    )
    out = pipe.answer('For an EU261-covered non-intra-EU 4000 km flight cancelled one day before departure, with no extraordinary circumstances, I accept rerouting that arrives three hours late. What compensation amount and care apply?',
                      memory=mem, profile=prof, user_id="demo_user", expand_references=True)
    print(out["answer"])
    print("\nReferences:")
    for r in out["references"]:
        print("  ", r)
else:
    print("Set GEMINI_API_KEY to run the full generative pipeline.")

## 11 · Where to go next — citations, state of the art

What we built is **Level 1** citation: the model self-tags `[[n]]`, and we validate
the *number is real*. State-of-the-art adds **verification**:

- **Level 2** structured output — model returns `{sentence, support_ids, quote}`.
- **Level 3** quote-grounded spans — match the quote back into the source for exact
  offsets; drop unverifiable quotes.
- **Level 4** NLI / entailment (ALCE) — check the passage actually *entails* the
  claim; score citation **precision & recall**.

The retrieval stack (chunking → hybrid → fusion → re-rank) is where most answer
quality is won. 🎓 You built all of it.